# 📊 Interactive Portfolio Risk Analytics — VaR Calculator

This notebook computes **Value-at-Risk (VaR)** for a stock or portfolio using three methods:
- **Historical VaR** — based on actual past returns
- **Parametric (Gaussian) VaR** — assumes normally distributed returns
- **Monte Carlo VaR** — simulates thousands of possible return paths

---

## 1. Install & Import Dependencies

In [ ]:
# Install required libraries (run once)
!pip install yfinance pandas numpy scipy matplotlib seaborn --quiet

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ All libraries loaded successfully.')

## 2. Configuration — Set Your Portfolio Here

In [ ]:
# ─── USER CONFIGURATION ───────────────────────────────────────────

# Single ticker OR a list of tickers for a portfolio
TICKERS = ['AAPL', 'MSFT', 'GOOGL']       # e.g. ['SPY'] or ['AAPL','MSFT','TSLA']

# Portfolio weights (must sum to 1.0). Use None for equal weighting.
WEIGHTS = None                              # e.g. [0.5, 0.3, 0.2] or None

# Date range for historical data
START_DATE = '2020-01-01'
END_DATE   = '2024-12-31'

# Risk parameters
CONFIDENCE_LEVEL  = 0.95     # 0.90, 0.95, or 0.99
PORTFOLIO_VALUE   = 100_000  # USD
HOLDING_PERIOD    = 1        # Days

# Monte Carlo settings
MC_SIMULATIONS = 10_000
MC_HORIZON     = 1           # Days ahead to simulate

# ──────────────────────────────────────────────────────────────────
print(f'Portfolio : {TICKERS}')
print(f'Period    : {START_DATE} → {END_DATE}')
print(f'Confidence: {CONFIDENCE_LEVEL*100:.0f}%')
print(f'Portfolio : ${PORTFOLIO_VALUE:,.0f}')

## 3. Download Historical Market Data

In [ ]:
def download_data(tickers, start, end):
    """Download adjusted close prices from Yahoo Finance."""
    print(f'Downloading data for: {tickers} ...')
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    
    if isinstance(tickers, list) and len(tickers) > 1:
        prices = raw['Close']
    else:
        prices = raw[['Close']]
        prices.columns = tickers if isinstance(tickers, list) else [tickers]
    
    prices.dropna(how='all', inplace=True)
    print(f'✅ Downloaded {len(prices)} trading days for {list(prices.columns)}')
    return prices


prices = download_data(TICKERS, START_DATE, END_DATE)
prices.tail()

## 4. Compute Portfolio Returns

In [ ]:
def compute_portfolio_returns(prices, weights=None):
    """Compute weighted daily log-returns for the portfolio."""
    returns = np.log(prices / prices.shift(1)).dropna()
    
    n = returns.shape[1]
    w = np.array(weights) if weights else np.ones(n) / n
    assert abs(w.sum() - 1.0) < 1e-6, 'Weights must sum to 1.0'
    
    portfolio_returns = returns @ w  # weighted sum
    
    print(f'Individual weights : {dict(zip(returns.columns, w.round(4)))}')
    print(f'Return observations: {len(portfolio_returns)}')
    print(f'Mean daily return  : {portfolio_returns.mean():.4%}')
    print(f'Daily volatility   : {portfolio_returns.std():.4%}')
    return portfolio_returns, returns, w


port_returns, ind_returns, weights_arr = compute_portfolio_returns(prices, WEIGHTS)

## 5. VaR Models

In [ ]:
alpha = 1 - CONFIDENCE_LEVEL  # tail probability

# ── 5a. Historical VaR ──────────────────────────────────────────
def historical_var(returns, alpha, portfolio_value):
    """Non-parametric VaR from empirical return distribution."""
    var_pct = np.percentile(returns, alpha * 100)
    var_usd = abs(var_pct) * portfolio_value
    cvar_pct = returns[returns <= var_pct].mean()   # Expected Shortfall
    cvar_usd = abs(cvar_pct) * portfolio_value
    return var_pct, var_usd, cvar_pct, cvar_usd


# ── 5b. Parametric (Gaussian) VaR ───────────────────────────────
def parametric_var(returns, alpha, portfolio_value):
    """VaR assuming normally distributed returns."""
    mu    = returns.mean()
    sigma = returns.std()
    z     = stats.norm.ppf(alpha)
    var_pct  = mu + z * sigma
    var_usd  = abs(var_pct) * portfolio_value
    cvar_pct = mu - sigma * stats.norm.pdf(z) / alpha
    cvar_usd = abs(cvar_pct) * portfolio_value
    return var_pct, var_usd, cvar_pct, cvar_usd, mu, sigma


# ── 5c. Monte Carlo VaR ─────────────────────────────────────────
def monte_carlo_var(returns, alpha, portfolio_value, n_sims, horizon):
    """VaR via Monte Carlo simulation of GBM paths."""
    mu    = returns.mean()
    sigma = returns.std()
    np.random.seed(42)
    simulated = np.random.normal(mu * horizon, sigma * np.sqrt(horizon), n_sims)
    var_pct   = np.percentile(simulated, alpha * 100)
    var_usd   = abs(var_pct) * portfolio_value
    cvar_pct  = simulated[simulated <= var_pct].mean()
    cvar_usd  = abs(cvar_pct) * portfolio_value
    return var_pct, var_usd, cvar_pct, cvar_usd, simulated


# ── Run all three ────────────────────────────────────────────────
hist_var, hist_var_usd, hist_cvar, hist_cvar_usd = historical_var(port_returns, alpha, PORTFOLIO_VALUE)
para_var, para_var_usd, para_cvar, para_cvar_usd, mu, sigma = parametric_var(port_returns, alpha, PORTFOLIO_VALUE)
mc_var, mc_var_usd, mc_cvar, mc_cvar_usd, mc_sims = monte_carlo_var(port_returns, alpha, PORTFOLIO_VALUE, MC_SIMULATIONS, MC_HORIZON)

print('\n' + '='*55)
print(f'  PORTFOLIO VaR RESULTS @ {CONFIDENCE_LEVEL*100:.0f}% Confidence')
print('='*55)
print(f'  {'Method':<22} {'VaR %':>8}  {'VaR $':>12}  {'CVaR $':>12}')
print('-'*55)
print(f'  {'Historical':<22} {hist_var:>8.3%}  ${hist_var_usd:>11,.0f}  ${hist_cvar_usd:>11,.0f}')
print(f'  {'Parametric (Gaussian)':<22} {para_var:>8.3%}  ${para_var_usd:>11,.0f}  ${para_cvar_usd:>11,.0f}')
print(f'  {'Monte Carlo':<22} {mc_var:>8.3%}  ${mc_var_usd:>11,.0f}  ${mc_cvar_usd:>11,.0f}')
print('='*55)

## 6. Visualisations

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.3)

COLORS = {'hist': '#e74c3c', 'para': '#3498db', 'mc': '#2ecc71', 'neutral': '#95a5a6'}

# ── Plot 1: Cumulative price chart ──────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
norm_prices = prices / prices.iloc[0] * 100
for col in norm_prices.columns:
    ax1.plot(norm_prices.index, norm_prices[col], linewidth=1.8, label=col)
ax1.set_title('Normalised Price History (Base = 100)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Indexed Price')
ax1.legend()
ax1.set_xlabel('')

# ── Plot 2: Return distribution + all VaR lines ─────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(port_returns, bins=80, density=True, color='#7f8c8d', alpha=0.5, label='Empirical returns')

# Overlay normal fit
x = np.linspace(port_returns.min(), port_returns.max(), 400)
ax2.plot(x, stats.norm.pdf(x, mu, sigma), color=COLORS['para'], lw=2, label='Normal fit')

# VaR threshold lines
ax2.axvline(hist_var, color=COLORS['hist'], lw=2, linestyle='--', label=f'Hist VaR {hist_var:.2%}')
ax2.axvline(para_var, color=COLORS['para'], lw=2, linestyle=':',  label=f'Para VaR {para_var:.2%}')
ax2.axvline(mc_var,   color=COLORS['mc'],   lw=2, linestyle='-.',  label=f'MC VaR   {mc_var:.2%}')

ax2.set_title(f'Portfolio Return Distribution & VaR Thresholds\n({CONFIDENCE_LEVEL*100:.0f}% Confidence)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Daily Log-Return')
ax2.set_ylabel('Density')
ax2.legend(fontsize=8)

# ── Plot 3: Monte Carlo loss distribution ───────────────────────
ax3 = fig.add_subplot(gs[1, 0])
losses_usd = -mc_sims * PORTFOLIO_VALUE
ax3.hist(losses_usd, bins=100, color=COLORS['mc'], alpha=0.6, label='Simulated P&L')
ax3.axvline(mc_var_usd, color='black', lw=2.5, linestyle='--', label=f'MC VaR = ${mc_var_usd:,.0f}')
ax3.axvline(mc_cvar_usd, color='orange', lw=2, linestyle=':', label=f'CVaR = ${mc_cvar_usd:,.0f}')
ax3.set_title(f'Monte Carlo Simulated Losses ({MC_SIMULATIONS:,} runs)', fontsize=13, fontweight='bold')
ax3.set_xlabel('Loss ($)')
ax3.set_ylabel('Frequency')
ax3.legend()

# ── Plot 4: VaR comparison bar chart ────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
methods = ['Historical', 'Parametric', 'Monte Carlo']
var_vals  = [hist_var_usd,  para_var_usd,  mc_var_usd]
cvar_vals = [hist_cvar_usd, para_cvar_usd, mc_cvar_usd]

x_pos = np.arange(len(methods))
bar_w = 0.35
b1 = ax4.bar(x_pos - bar_w/2, var_vals,  bar_w, label='VaR',  color=[COLORS['hist'], COLORS['para'], COLORS['mc']], alpha=0.85)
b2 = ax4.bar(x_pos + bar_w/2, cvar_vals, bar_w, label='CVaR', color=[COLORS['hist'], COLORS['para'], COLORS['mc']], alpha=0.45)

ax4.set_xticks(x_pos)
ax4.set_xticklabels(methods)
ax4.set_ylabel('Dollar Loss ($)')
ax4.set_title(f'VaR vs CVaR by Method\n(${PORTFOLIO_VALUE:,.0f} portfolio, {CONFIDENCE_LEVEL*100:.0f}% confidence)', fontsize=13, fontweight='bold')
ax4.legend()
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

# Value labels on bars
for bar in b1:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8)

fig.suptitle(f'Portfolio Risk Analytics — {" | ".join(TICKERS)}', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('/tmp/var_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard rendered.')

## 7. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Method'          : ['Historical', 'Parametric (Gaussian)', 'Monte Carlo'],
    'VaR (%)' : [f'{hist_var:.3%}', f'{para_var:.3%}', f'{mc_var:.3%}'],
    'VaR ($)' : [f'${hist_var_usd:,.0f}', f'${para_var_usd:,.0f}', f'${mc_var_usd:,.0f}'],
    'CVaR / ES ($)': [f'${hist_cvar_usd:,.0f}', f'${para_cvar_usd:,.0f}', f'${mc_cvar_usd:,.0f}'],
}).set_index('Method')

print(f'\nPortfolio : {TICKERS}')
print(f'Window    : {START_DATE} → {END_DATE}')
print(f'Confidence: {CONFIDENCE_LEVEL*100:.0f}%  |  Portfolio Value: ${PORTFOLIO_VALUE:,.0f}\n')
summary

---
### 📝 Interpretation Guide

| Term | Meaning |
|------|----------|
| **VaR (%)** | Maximum expected 1-day loss (as % of portfolio) at the chosen confidence level |
| **VaR ($)** | That loss in dollar terms |
| **CVaR / ES** | *Conditional* VaR — the average loss *beyond* the VaR threshold (tail-risk) |

> **Example read:** *"At 95% confidence, we do not expect to lose more than \$X in a single day. On the worst 5% of days, the average loss is \$Y (CVaR)."*

**Next steps for the full dashboard project:**
1. Wrap this logic in a **Streamlit** or **Dash** web app
2. Add a **ticker input widget** and **confidence level slider**
3. Deploy to **Streamlit Cloud** or **Heroku** for a live URL